In [1]:
import hashlib
import json
import os
import time
from pathlib import Path
from typing import Dict, Any, Optional, Tuple

from repository_tree_sitter_scanner import RepositoryScanner

# ── Config ───────────────────────────────────────────────────────────
REPO_PATH   = Path("../test").resolve()          # repo root to scan
CACHE_DIR   = REPO_PATH / ".cache"
CACHE_FILE  = CACHE_DIR / "repo_tree.json"
CACHE_META  = CACHE_DIR / "repo_tree.meta.json"

# Extensions we care about when hashing (must match the scanner)
TRACKED_EXTS = {".py", ".pyi", ".cpp", ".cc", ".cxx",
                ".hpp", ".hh", ".hxx", ".h", ".c",
                ".rs"}

# Directories that the scanner skips — we skip them when hashing too
EXCLUDE_DIRS = {".git", ".venv", "venv", "__pycache__",
                "node_modules", "target", "build", "dist",
                ".mypy_cache", ".pytest_cache", ".tox",
                ".cache"}

In [2]:
def _iter_tracked_files(root: Path):
    """Yield every source file the scanner would touch."""
    for dirpath, dirnames, filenames in os.walk(root, topdown=True):
        dirnames[:] = [d for d in dirnames if d not in EXCLUDE_DIRS]
        for name in filenames:
            p = Path(dirpath) / name
            if p.suffix.lower() in TRACKED_EXTS:
                yield p


def _hash_file_fast(path: Path) -> str:
    """Cheap fingerprint: path + size + mtime."""
    st = path.stat()
    h = hashlib.blake2b(digest_size=16)
    h.update(str(path).encode())
    h.update(str(st.st_size).encode())
    h.update(str(int(st.st_mtime_ns)).encode())
    return h.hexdigest()


# Full-content hash (slower, more robust) — swap in if you want:
# def _hash_file_fast(path: Path) -> str:
#     h = hashlib.blake2b(digest_size=16)
#     with path.open("rb") as fh:
#         for chunk in iter(lambda: fh.read(1 << 20), b""):
#             h.update(chunk)
#     return h.hexdigest()


def compute_repo_fingerprint(root: Path) -> Tuple[str, int]:
    """
    Return (aggregate_hash, file_count).

    The aggregate hash is order-independent (files are sorted first) and
    includes the path, so renames invalidate the cache.
    """
    entries = []
    for p in _iter_tracked_files(root):
        rel = p.relative_to(root).as_posix()
        entries.append(f"{rel}:{_hash_file_fast(p)}")

    entries.sort()
    agg = hashlib.blake2b(digest_size=16)
    for e in entries:
        agg.update(e.encode())
        agg.update(b"\n")
    return agg.hexdigest(), len(entries)

In [3]:
def load_cache() -> Optional[Dict[str, Any]]:
    """Return the cached scan dict, or None if the cache is unusable."""
    if not CACHE_FILE.exists() or not CACHE_META.exists():
        return None
    try:
        meta = json.loads(CACHE_META.read_text(encoding="utf-8"))
        tree = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        print(f"[cache] unreadable, will rebuild ({exc})")
        return None
    return {"meta": meta, "tree": tree}


def save_cache(tree: Dict[str, Any], fingerprint: str, file_count: int) -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    meta = {
        "fingerprint": fingerprint,
        "file_count": file_count,
        "created_at": time.time(),
        "scanner_version": 1,   # bump this if the scanner output shape changes
    }
    # Atomic-ish write: temp file then replace
    tmp_tree = CACHE_FILE.with_suffix(".json.tmp")
    tmp_meta = CACHE_META.with_suffix(".json.tmp")
    tmp_tree.write_text(json.dumps(tree, indent=2), encoding="utf-8")
    tmp_meta.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    tmp_tree.replace(CACHE_FILE)
    tmp_meta.replace(CACHE_META)
    print(f"[cache] wrote {CACHE_FILE} "
          f"({CACHE_FILE.stat().st_size / 1024:.1f} KB)")

In [4]:
def get_repo_tree(force_refresh: bool = False) -> Dict[str, Any]:
    """
    Return the full repo tree as a dict.

    - If the cache is fresh, load it (near-instant).
    - Otherwise scan with Tree-sitter and write the cache.
    - `force_refresh=True` bypasses the cache unconditionally.
    """
    t0 = time.time()
    fingerprint, file_count = compute_repo_fingerprint(REPO_PATH)

    if not force_refresh:
        cached = load_cache()
        if cached and cached["meta"].get("fingerprint") == fingerprint:
            print(f"[cache] HIT  ({file_count} files, "
                  f"loaded in {time.time() - t0:.3f}s)")
            return cached["tree"]
        if cached:
            print("[cache] MISS (repo changed since last scan)")
        else:
            print("[cache] MISS (no cache on disk)")
    else:
        print("[cache] forced refresh")

    # ── Build fresh ──────────────────────────────────────────────────
    scanner = RepositoryScanner()
    result = scanner.scan(REPO_PATH)
    tree = result.to_dict()

    save_cache(tree, fingerprint, file_count)
    print(f"[cache] scan + write done in {time.time() - t0:.3f}s "
          f"({result.stats.get('total_files', 0)} files, "
          f"{result.stats.get('total_nodes', 0)} nodes)")
    return tree

In [6]:
tree = get_repo_tree()

print()
print("root_path      :", tree["root_path"])
print("stats          :", tree["stats"])

# Show the first few top-level files
for f in tree["files"][:5]:
    err = f.get("error")
    tag = f"ERR({err})" if err else "ok"
    print(f"  [{f['language']:6s}] {f['path']}  {tag}")

[cache] HIT  (12 files, loaded in 0.007s)

root_path      : C:\Projects\Harness\test
stats          : {'total_files': 12, 'files_by_language': {'cpp': 12}, 'total_nodes': 732, 'errors': 0}
  [cpp   ] Cpp-Tetris-Game-with-raylib-main\src\block.cpp  ok
  [cpp   ] Cpp-Tetris-Game-with-raylib-main\src\block.h  ok
  [cpp   ] Cpp-Tetris-Game-with-raylib-main\src\blocks.cpp  ok
  [cpp   ] Cpp-Tetris-Game-with-raylib-main\src\colors.cpp  ok
  [cpp   ] Cpp-Tetris-Game-with-raylib-main\src\colors.h  ok


In [7]:
# Rebuild the cache from scratch (e.g. after tweaking node_types)
tree = get_repo_tree(force_refresh=True)

[cache] forced refresh
[cache] wrote C:\Projects\Harness\test\.cache\repo_tree.json (507.6 KB)
[cache] scan + write done in 0.017s (12 files, 732 nodes)


In [11]:
import functions
tools_schema = functions.TOOLS

def dispatch(tool_call):
    return functions.call_tool(tool_call.name, tool_call.arguments)